# Build IR Object-Detection Dataset (Colab GPU)

Runs the 4-stage pipeline (`sample → auto-annotate → AI-assisted review → build`) on a GPU.
Expects the repo + Drive&Act `data/` and `activities_3s/` available under `DATA_ROOT`.

**Not fully autonomous:** stages 2–3 propose labels; the human-validation gate in `README.md` accepts them before training.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point at the repo checkout that contains ir_object_detection/, data/, activities_3s/
import os
REPO = '/content/drive/MyDrive/DriveGuard/driveguard'  # <-- adjust
os.environ['DATA_ROOT'] = REPO
%cd $REPO

In [ ]:
!pip install -q -r requirements.txt -r ir_object_detection/requirements.txt
# VLM review (optional). Set your key, or run stage 3 with --no-vlm.
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # <-- paste key for the Claude vision review pass

## 2. Stage 1 — sample ~500 IR frames

In [ ]:
!python ir_object_detection/1_sample_ir_frames.py --target 500

## 3. Stage 2 — GroundingDINO + SAM2 auto-annotation
Preview on 10 frames first to tune thresholds in `config.py`.

In [ ]:
!python ir_object_detection/2_auto_annotate.py --limit 10   # preview
# !python ir_object_detection/2_auto_annotate.py            # full run

## 4. Stage 3 — AI-assisted review (heuristics + Claude vision)

In [ ]:
!python ir_object_detection/3_review_refine.py            # add --no-vlm to skip the Claude pass

## 5. Stage 4 — build dataset + QC report

In [ ]:
!python ir_object_detection/4_build_dataset.py
print(open('ir_object_detection/dataset/qc_report.md').read())

## 6. Format smoke test (1 epoch) — confirms data.yaml/labels are valid

In [ ]:
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='ir_object_detection/dataset/data.yaml', epochs=1, imgsz=640)
print('Format OK — now complete the human-validation gate in README.md before real training.')